# SQL JOINs
Combinación de tablas relacionales mediante claves compartidas.

In [1]:
import duckdb

# Crear conexión (base en memoria)
con = duckdb.connect()

# Crear tablas
con.execute("""
CREATE TABLE customers (
    customer_id INTEGER,
    name VARCHAR,
    city VARCHAR
);
""")

con.execute("""
CREATE TABLE transactions (
    transaction_id INTEGER,
    customer_id INTEGER,
    amount DECIMAL(10,2),
    date DATE
);
""")

# Insertar datos
con.execute("""
INSERT INTO customers VALUES
(1, 'Ana', 'Bogotá'),
(2, 'Luis', 'Medellín'),
(3, 'Pedro', 'Cali'),
(4, 'Sofía', 'Barranquilla');
""")

con.execute("""
INSERT INTO transactions VALUES
(101, 2, 150.00, '2024-01-10'),
(102, 3, 200.00, '2024-01-11'),
(103, 5, 300.00, '2024-01-12');
""")

In [2]:
#imprimir tablas como un dataframe
print("Customers:")
print(con.execute("SELECT * FROM customers").fetchdf())
print("\nTransactions:")
print(con.execute("SELECT * FROM transactions").fetchdf())

Customers:
   customer_id   name          city
0            1    Ana        Bogotá
1            2   Luis      Medellín
2            3  Pedro          Cali
3            4  Sofía  Barranquilla

Transactions:
   transaction_id  customer_id  amount       date
0             101            2   150.0 2024-01-10
1             102            3   200.0 2024-01-11
2             103            5   300.0 2024-01-12


In [3]:
# INNER JOIN
# El inner join devuelve filas cuando hay una coincidencia en ambas tablas.
# Si no hay coincidencia, la fila no se incluye en el resultado.
con.execute("""
SELECT *
FROM customers
INNER JOIN transactions
ON customers.customer_id = transactions.customer_id;
""")
df_inner_join = con.fetchdf()
df_inner_join

,customer_id,name,city,transaction_id,customer_id_1,amount,date
0,2,Luis,Medellín,101,2,150.0,2024-01-10
1,3,Pedro,Cali,102,3,200.0,2024-01-11


In [4]:

# LEFT JOIN
# El left join devuelve todas las filas de la tabla de la izquierda (customers)
# y las filas coincidentes de la tabla de la derecha (transactions). Si no hay coincidencia,
# las columnas de la tabla de la derecha tendrán valores NULL.
con.execute("""
SELECT *
FROM customers
LEFT JOIN transactions
ON customers.customer_id = transactions.customer_id;
""")

df_left_join = con.fetchdf()
df_inner_join

,customer_id,name,city,transaction_id,customer_id_1,amount,date
0,2,Luis,Medellín,101,2,150.0,2024-01-10
1,3,Pedro,Cali,102,3,200.0,2024-01-11


In [5]:
# RIGHT JOIN
# El right join devuelve todas las filas de la tabla de la derecha (transactions)
# y las filas coincidentes de la tabla de la izquierda (customers). Si no hay coincidencia,
# las columnas de la tabla de la izquierda tendrán valores NULL.
con.execute("""
SELECT *
FROM customers
RIGHT JOIN transactions
ON customers.customer_id = transactions.customer_id;
""")
df_right_join = con.fetchdf()
df_right_join



,customer_id,name,city,transaction_id,customer_id_1,amount,date
0,2,Luis,Medellín,101,2,150.0,2024-01-10
1,3,Pedro,Cali,102,3,200.0,2024-01-11
2,<NA>,NaN,NaN,103,5,300.0,2024-01-12


In [6]:
# FULL OUTER JOIN
# El full outer join devuelve todas las filas cuando hay una coincidencia en una de las tablas.
# Si no hay coincidencia, las columnas de la tabla sin coincidencia tendrán valores NULL.
con.execute("""
SELECT *
FROM customers
FULL OUTER JOIN transactions
ON customers.customer_id = transactions.customer_id;
""")
df_full_outer_join = con.fetchdf()
df_full_outer_join

,customer_id,name,city,transaction_id,customer_id_1,amount,date
0,2,Luis,Medellín,101,2,150.0,2024-01-10
1,3,Pedro,Cali,102,3,200.0,2024-01-11
2,1,Ana,Bogotá,<NA>,<NA>,NaN,NaT
3,4,Sofía,Barranquilla,<NA>,<NA>,NaN,NaT
4,<NA>,NaN,NaN,103,5,300.0,2024-01-12


In [7]:
# CROSS JOIN
# El cross join devuelve el producto cartesiano de las dos tablas, es decir, todas las
# combinaciones posibles de filas entre las dos tablas.
con.execute("""
SELECT *
FROM customers
CROSS JOIN transactions;
""")
df_cross_join = con.fetchdf()
df_cross_join

,customer_id,name,city,transaction_id,customer_id_1,amount,date
0,1,Ana,Bogotá,101,2,150.0,2024-01-10
1,2,Luis,Medellín,101,2,150.0,2024-01-10
2,3,Pedro,Cali,101,2,150.0,2024-01-10
3,4,Sofía,Barranquilla,101,2,150.0,2024-01-10
4,1,Ana,Bogotá,102,3,200.0,2024-01-11
5,2,Luis,Medellín,102,3,200.0,2024-01-11
6,3,Pedro,Cali,102,3,200.0,2024-01-11
7,4,Sofía,Barranquilla,102,3,200.0,2024-01-11
8,1,Ana,Bogotá,103,5,300.0,2024-01-12
9,2,Luis,Medellín,103,5,300.0,2024-01-12


# GROUP BY
Agrupa filas con valores iguales para aplicar funciones de agregación como `COUNT`, `SUM`, `AVG`, etc.

In [8]:
con.execute("""
INSERT INTO transactions VALUES
(104, 2, 80.00, '2024-01-15'),
(105, 2, 120.00, '2024-01-20'),
(106, 3, 50.00, '2024-01-22'),
(107, 1, 300.00, '2024-01-25'),
(108, 1, 100.00, '2024-01-28'),
(109, 4, 75.00, '2024-02-01');
""")

# imprimir tabla como un dataframe
print("\nTransactions after inserting more data:")
df_transactions=(con.execute("SELECT * FROM transactions").fetchdf())
df_transactions


Transactions after inserting more data:


,transaction_id,customer_id,amount,date
0,101,2,150.0,2024-01-10
1,102,3,200.0,2024-01-11
2,103,5,300.0,2024-01-12
3,104,2,80.0,2024-01-15
4,105,2,120.0,2024-01-20
5,106,3,50.0,2024-01-22
6,107,1,300.0,2024-01-25
7,108,1,100.0,2024-01-28
8,109,4,75.0,2024-02-01


In [9]:
# Grup BY
con.execute("""
SELECT customer_id, COUNT(*) AS num_transactions, SUM(amount) AS total_amount
FROM transactions
GROUP BY customer_id;
""")
df_group_by = con.fetchdf()
df_group_by

,customer_id,num_transactions,total_amount
0,1,2,400.0
1,2,3,350.0
2,3,2,250.0
3,4,1,75.0
4,5,1,300.0


# ejercicio group by

In [10]:
con.execute("""
INSERT INTO customers VALUES
(6, 'Carlos', 'Bogotá'),
(7, 'Laura', 'Medellín'),
(8, 'María', 'Cali'),
(9, 'Andrés', 'Bogotá');
""")
con.execute("""
INSERT INTO transactions VALUES
(110, 6, 500.00, '2024-02-05'),
(111, 6, 150.00, '2024-02-07'),
(112, 7, 250.00, '2024-02-08'),
(113, 8, 300.00, '2024-02-10'),
(114, 9, 50.00, '2024-02-12'),
(115, 9, 70.00, '2024-02-14');
""")

Perfecto 😎 vamos a hacerlo tipo parcial.

---

## 🎯 Ejercicio 1 (nivel medio)

Usando las tablas `customers` y `transactions`:

👉 Muestra:

* `city`
* Cantidad total de transacciones por ciudad
* Total de dinero gastado por ciudad

📌 Reglas:

* Debes usar `JOIN`
* Debes usar `GROUP BY`
* No puedes usar subconsultas

---

### 🔎 Lo que debes pensar

1. Necesitas la ciudad → está en `customers`
2. Necesitas el amount → está en `transactions`
3. Debes unirlas por `customer_id`
4. Luego agrupar por `city`

---



In [22]:
#jjoin
con.execute("""
select city, count(transaction_id) as transaction_count, sum(amount) as total_amount
from customers
join transactions
on customers.customer_id = transactions.customer_id
group by city;

""")
df_join = con.fetchdf()
df_join

,city,transaction_count,total_amount
0,Cali,3,550.0
1,Medellín,4,600.0
2,Bogotá,6,1170.0
3,Barranquilla,1,75.0
